# Chuyển `customer_turns` thành audio bằng FPT.AI TTS

Notebook Google Colab này đọc **một file scenario JSON**, lấy `calls.<call_id>.customer_turns`, rồi tạo **một file audio cho mỗi cuộc gọi** bằng API FPT.AI tương thích OpenAI.

Trước khi chạy:

1. Chạy ô cài đặt thư viện và mount Google Drive.
2. Lưu API key với tên `FPT_API_KEY` trong **Colab Secrets**, hoặc thêm `FPT_API_KEY=...` vào `/content/.env`. Không ghi key trực tiếp vào notebook.
3. Chọn `SCENARIO_FILE`, kiểm tra phần xem trước, rồi đổi `RUN_TTS = True`.

> Mặc định `RUN_TTS = False` để Run All không vô tình sử dụng quota. Notebook dùng `base_url=https://token-api.fpt.ai`, model `FPT.AI-VITs` và phương thức `client.audio.speech.create(...)` theo mẫu tích hợp được cung cấp.

In [ ]:
%pip install -q openai python-dotenv

In [ ]:
import json
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Audio, display
from openai import OpenAI

## 1. Mount Google Drive và chọn file đầu vào

Đổi `SCENARIO_FILE` nếu file của bạn có tên hoặc vị trí khác. Các đường dẫn phải là đối tượng `Path`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content")
DATASET_DIR = Path("/content/drive/MyDrive/scenarios_complete dataset")

# Thay tên file tại đây nếu cần.
SCENARIO_FILE = DATASET_DIR / "SC-001.json"
OUTPUT_DIR = Path("/content")

if not SCENARIO_FILE.is_file():
    raise FileNotFoundError(f"Không tìm thấy file: {SCENARIO_FILE}")

print("Input :", SCENARIO_FILE)
print("Output:", OUTPUT_DIR)

## 2. Đọc và kiểm tra `customer_turns`

In [ ]:
def load_customer_turns(json_path: Path) -> tuple[str, dict[str, list[str]]]:
    """Đọc scenario và trả về (scenario_id, {call_id: customer_turns})."""
    with json_path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    if not isinstance(payload, dict):
        raise ValueError("Nội dung JSON phải là một object.")

    scenario_id = str(payload.get("scenario_id") or json_path.stem)
    calls = payload.get("calls")
    if not isinstance(calls, dict) or not calls:
        raise ValueError("Không tìm thấy object 'calls' hoặc 'calls' đang rỗng.")

    turns_by_call: dict[str, list[str]] = {}
    for call_id, call_data in calls.items():
        if not isinstance(call_data, dict):
            raise ValueError(f"{call_id} phải là một object.")

        turns = call_data.get("customer_turns")
        if not isinstance(turns, list):
            raise ValueError(f"{call_id}.customer_turns phải là một list.")
        if not all(isinstance(turn, str) for turn in turns):
            raise ValueError(f"Mọi phần tử trong {call_id}.customer_turns phải là chuỗi.")

        cleaned_turns = [turn.strip() for turn in turns if turn.strip()]
        if not cleaned_turns:
            raise ValueError(f"{call_id}.customer_turns không có câu hợp lệ.")
        turns_by_call[str(call_id)] = cleaned_turns

    return scenario_id, turns_by_call


scenario_id, customer_turns_by_call = load_customer_turns(SCENARIO_FILE)
print(f"Scenario: {scenario_id}")
for call_id, turns in customer_turns_by_call.items():
    print(f"\n{call_id} ({len(turns)} câu):")
    for index, text in enumerate(turns, start=1):
        print(f"  {index}. {text}")

## 3. Cấu hình FPT.AI TTS

- `CALL_IDS = None`: xử lý tất cả các cuộc gọi; ví dụ `["call_1"]` để chỉ xử lý một call.
- `banmai` là giọng nữ miền Bắc. Một số voice FPT khác: `lannhi`, `leminh`, `myan`, `thuminh`, `giahuy`, `linhsan`.
- Mỗi call được ghép thành một chuỗi, với hai dòng trống giữa các lượt khách.
- `OVERWRITE = False`: bỏ qua file đã tồn tại.

In [ ]:
BASE_URL = "https://token-api.fpt.ai"
MODEL = "FPT.AI-VITs"
VOICE = "banmai"
RESPONSE_FORMAT = "mp3"
SPEED = 1.0
MAX_CHARACTERS = 5_000

CALL_IDS = None  # None = tất cả; ví dụ: ["call_1"]
OVERWRITE = False
RUN_TTS = False  # Đổi thành True sau khi đã kiểm tra nội dung phía trên.

load_dotenv(PROJECT_ROOT / ".env")
FPT_API_KEY = os.getenv("FPT_API_KEY") or os.getenv("FPT_AI_API_KEY")

# Nếu chưa có trong biến môi trường/.env, thử lấy từ Colab Secrets.
if not FPT_API_KEY:
    try:
        from google.colab import userdata

        FPT_API_KEY = userdata.get("FPT_API_KEY")
    except Exception:
        FPT_API_KEY = None

print("Đã tìm thấy FPT_API_KEY:", bool(FPT_API_KEY))

In [ ]:
def safe_filename(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._") or "audio"


def audio_extension(response_format: str) -> str:
    normalized = response_format.lower().strip()
    supported = {"mp3", "wav", "opus", "aac", "flac", "pcm"}
    if normalized not in supported:
        raise ValueError(f"RESPONSE_FORMAT không được hỗ trợ: {response_format}")
    return normalized


def build_call_text(text_chunks: list[str]) -> str:
    text = "\n\n".join(chunk.strip() for chunk in text_chunks if chunk.strip())
    if len(text) < 3:
        raise ValueError("Nội dung TTS phải có ít nhất 3 ký tự.")
    if len(text) > MAX_CHARACTERS:
        raise ValueError(
            f"Nội dung có {len(text):,} ký tự; giới hạn hiện tại là {MAX_CHARACTERS:,}."
        )
    return text


def synthesize_call(
    client: OpenAI,
    scenario_id: str,
    call_id: str,
    text_chunks: list[str],
    output_dir: Path,
) -> tuple[Path, int, dict]:
    """Gọi FPT.AI TTS và tạo một file audio cho một call."""
    text = build_call_text(text_chunks)
    extension = audio_extension(RESPONSE_FORMAT)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / (
        f"{safe_filename(scenario_id)}_{safe_filename(call_id)}_fpt_customer.{extension}"
    )

    response = client.audio.speech.create(
        model=MODEL,
        voice=VOICE,
        input=text,
        response_format=RESPONSE_FORMAT,
        speed=SPEED,
    )
    response.write_to_file(str(output_path))

    if not output_path.is_file():
        raise RuntimeError(f"FPT.AI không tạo file: {output_path}")
    byte_count = output_path.stat().st_size
    if byte_count == 0:
        raise RuntimeError("FPT.AI trả về file audio rỗng.")

    metadata = {
        "text_characters": len(text),
        "model": MODEL,
        "voice": VOICE,
        "speed": SPEED,
    }
    return output_path, byte_count, metadata

## 4. Gọi FPT.AI và lưu audio

Ô này chỉ gọi API khi `RUN_TTS = True`. Kết quả được lưu trong `/content/<scenario_id>/`, kèm `manifest_fpt.json`.

In [ ]:
if not RUN_TTS:
    print("Chưa gọi API. Hãy đổi RUN_TTS = True trong ô cấu hình khi sẵn sàng.")
else:
    if not FPT_API_KEY:
        raise EnvironmentError(
            "Thiếu FPT_API_KEY trong Colab Secrets, /content/.env hoặc biến môi trường."
        )

    if CALL_IDS is None:
        selected_call_ids = list(customer_turns_by_call)
    else:
        if isinstance(CALL_IDS, str):
            raise TypeError("CALL_IDS phải là list, ví dụ ['call_1'], không phải chuỗi.")
        selected_call_ids = list(dict.fromkeys(CALL_IDS))
        missing = [call_id for call_id in selected_call_ids if call_id not in customer_turns_by_call]
        if missing:
            raise KeyError(f"CALL_IDS không tồn tại trong file: {missing}")

    extension = audio_extension(RESPONSE_FORMAT)
    scenario_output_dir = OUTPUT_DIR / safe_filename(scenario_id)
    results = []
    generated_files = []
    client = OpenAI(api_key=FPT_API_KEY, base_url=BASE_URL)

    try:
        for call_id in selected_call_ids:
            expected_path = scenario_output_dir / (
                f"{safe_filename(scenario_id)}_{safe_filename(call_id)}_fpt_customer.{extension}"
            )
            if expected_path.exists() and not OVERWRITE:
                print(f"Bỏ qua file đã tồn tại: {expected_path.name}")
                results.append(
                    {
                        "call_id": call_id,
                        "status": "skipped_existing",
                        "output_file": str(expected_path),
                    }
                )
                continue

            try:
                output_path, byte_count, metadata = synthesize_call(
                    client=client,
                    scenario_id=scenario_id,
                    call_id=call_id,
                    text_chunks=customer_turns_by_call[call_id],
                    output_dir=scenario_output_dir,
                )
                generated_files.append(output_path)
                results.append(
                    {
                        "call_id": call_id,
                        "status": "generated",
                        "turn_count": len(customer_turns_by_call[call_id]),
                        "bytes": byte_count,
                        "output_file": str(output_path),
                        **metadata,
                    }
                )
                print(f"Đã ghi {byte_count:,} bytes: {output_path}")
            except Exception as exc:
                results.append(
                    {
                        "call_id": call_id,
                        "status": "error",
                        "error": f"{type(exc).__name__}: {exc}",
                    }
                )
                print(f"Lỗi ở {call_id}: {type(exc).__name__}: {exc}")
    finally:
        client.close()

    scenario_output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = scenario_output_dir / "manifest_fpt.json"
    manifest = {
        "source_file": str(SCENARIO_FILE),
        "scenario_id": scenario_id,
        "base_url": BASE_URL,
        "model": MODEL,
        "voice": VOICE,
        "response_format": RESPONSE_FORMAT,
        "speed": SPEED,
        "results": results,
    }
    manifest_path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Manifest:", manifest_path)

    if generated_files:
        print("Nghe thử file vừa tạo đầu tiên:")
        display(Audio(filename=str(generated_files[0])))